# Text Language Detection

This notebook is intentionally minimal.

- Add one text string and see its detected language in its own output table.
- Or load a dataset (CSV) and see language results in a separate dataset table.

In [11]:
import pandas as pd
from langdetect import detect_langs

In [12]:
def detect_language(text: str):
    try:
        result = detect_langs(str(text))
        if not result:
            return "err", 0.0
        top = result[0]
        return top.lang, float(top.prob)
    except Exception:
        return "err", 0.0


def detect_language_in_dataframe(df: pd.DataFrame, text_column: str):
    out = df.copy()
    detected = out[text_column].apply(detect_language)
    out["detected_language"] = detected.apply(lambda x: x[0])
    out["confidence"] = detected.apply(lambda x: x[1])
    return out

In [13]:
LANGUAGE_NAME_MAP = {
    "es": "Spanish",
    "en": "English",
    "fr": "French",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "nl": "Dutch",
    "sv": "Swedish",
    "err": "Other",
}


def code_to_name(code: str):
    return LANGUAGE_NAME_MAP.get(code, "Other")


def format_percent(value: float):
    return f"{value:.1f}".replace(".", ",") + "%"


def print_language_percentage_report(
    result_df: pd.DataFrame,
    expected_percentages=None,
    title: str = "Language Detection %",
):
    expected_percentages = expected_percentages or {}
    total = len(result_df)
    if total == 0:
        print(f"----- {title} -----")
        print("No rows to analyze.")
        return

    names = result_df["detected_language"].apply(code_to_name)
    percentages = (names.value_counts(normalize=True) * 100).to_dict()

    print(f"----- {title} -----")
    used_keys = set()

    for lang_name, expected in expected_percentages.items():
        observed = percentages.get(lang_name, 0.0)
        used_keys.add(lang_name)
        print(f"{lang_name} -> {format_percent(observed)} ({round(expected)}% expected)")

    if expected_percentages:
        other_value = sum(v for k, v in percentages.items() if k not in used_keys)
        print(f"Other -> {format_percent(other_value)}")
    else:
        for lang_name, observed in sorted(percentages.items(), key=lambda x: x[1], reverse=True):
            print(f"{lang_name} -> {format_percent(observed)}")

In [14]:
# --- Option A: Detect one text ---
text_input = "Bonjour, je dois finir mon devoir ce soir."
lang, prob = detect_language(text_input)

single_text_result = pd.DataFrame([{
    "Input Text": text_input,
    "Detected Language": code_to_name(lang),
    "Confidence": f"{prob:.2%}",
}])

display(single_text_result)

,Input Text,Detected Language,Confidence
0,"Bonjour, je dois finir mon devoir ce soir.",French,100.00%


In [15]:
# --- Option B: Detect from a dataset (CSV) ---
# Set CSV_PATH to your dataset and TEXT_COLUMN to your text column name.
CSV_PATH = "Language Detection.csv"          # Example: "my_text_data.csv"
TEXT_COLUMN = "Text"   # Example column containing text

if CSV_PATH:
    df = pd.read_csv(CSV_PATH)
else:
    df = pd.DataFrame({
        "text": [
            "Hola, necesito ayuda con mi tarea.",
            "Este proyecto es para detectar idioma en texto.",
            "Necesito terminar el trabajo esta noche.",
            "This assignment is due tomorrow morning.",
            "Can you help me with the report?",
            "Je dois finir mon projet ce soir.",
            "Bonjour, comment allez-vous?",
            "Das ist ein sehr interessantes Projekt.",
            "Die Vorlesung beginnt um neun Uhr."
        ]
    })

result_df = detect_language_in_dataframe(df, TEXT_COLUMN)
dataset_result = result_df[[TEXT_COLUMN, "detected_language", "confidence"]].copy()
dataset_result = dataset_result.rename(columns={
    TEXT_COLUMN: "Dataset Text",
    "detected_language": "Detected Language",
    "confidence": "Confidence",
})

display(dataset_result.head(20))

# Optional expected percentages (edit these for your dataset)
EXPECTED_PERCENTAGES = {
    "Spanish": 66,
    "English": 16,
    "French": 9,
    "German": 9,
}

print()
print_language_percentage_report(result_df, expected_percentages=EXPECTED_PERCENTAGES, title="Langdetect %")

,Dataset Text,Detected Language,Confidence
0,"Nature, in the broadest sense, is the natural...",en,0.999998
1,"""Nature"" can refer to the phenomena of the phy...",en,0.999997
2,"The study of nature is a large, if not the onl...",en,0.999999
3,"Although humans are part of nature, human acti...",en,0.999996
4,[1] The word nature is borrowed from the Old F...,en,0.999997
5,"[2] In ancient philosophy, natura is mostly us...",en,0.999998
6,"[3][4] \nThe concept of nature as a whole, the...",en,0.999998
7,During the advent of modern scientific method ...,en,0.999995
8,"[5][6] With the Industrial revolution, nature ...",en,0.999997
9,"However, a vitalist vision of nature, closer t...",en,0.999998



----- Langdetect % -----
Spanish -> 7,5% (66% expected)
English -> 13,0% (16% expected)
French -> 9,7% (9% expected)
German -> 4,4% (9% expected)
Other -> 65,4%


In [16]:
# --- Interactive: detect a specific language by name or code ---
target_input = input("Enter language to detect (name e.g. Spanish or code e.g. es), or leave blank: " ).strip()
if target_input:
    inv_map = {v.lower(): k for k, v in LANGUAGE_NAME_MAP.items()}
    t = target_input.strip()
    if len(t) == 2:
        target_code = t.lower()
        target_name = code_to_name(target_code)
    else:
        target_code = inv_map.get(t.lower())
        if target_code is None:
            print(f"Unknown language name '{target_input}'. Try one of: {', '.join(sorted(set(LANGUAGE_NAME_MAP.values())))}")
            target_name = None
        else:
            target_name = code_to_name(target_code)
    if target_name:
        result_df['is_target'] = result_df['detected_language'].apply(lambda c: code_to_name(c).lower() == target_name.lower())
        matches = result_df[result_df['is_target']]
        print()
        print(f"Rows matching target language '{target_name}': {len(matches)} / {len(result_df)} ({format_percent(len(matches)/len(result_df)*100)})")
        display(matches[[TEXT_COLUMN, 'detected_language', 'confidence']].rename(columns={TEXT_COLUMN: 'Dataset Text', 'detected_language': 'Detected Language', 'confidence': 'Confidence'}).head(50))


Rows matching target language 'German': 457 / 10337 (4,4%)


,Dataset Text,Detected Language,Confidence
3000,Meu Deus.,de,0.857141
3044,Meu Deus.,de,0.857138
4150,attendez un instant.,de,0.714285
4344,"In september 2008 ontving Wikipedia de ""Quadri...",de,0.571920
6704,Pr.,de,0.999996
7570,Starò bene.,de,0.999996
8099,berbat ettim.,de,0.857140
8140,Affedersiniz?,de,0.714285
8648,kunde inte.,de,0.856242
9498,.Wir sind alle auf der Suche nach schnellen We...,de,0.999997
